# 📋 config.json 全走査

層種別・KV ヘッド・rope 上限・重み形状から KV を計算。


In [ ]:

def find_root():
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

import json

ROOT = find_root()
cfg = json.load(open(ROOT + "/config.json"))
tc = cfg.get("text_config") or {}

print("=== トップレベル ===")
for k in sorted(cfg):
    v = cfg[k]
    if isinstance(v, (dict, list)):
        print(f"  {k}: {type(v).__name__}({len(v)})")
    else:
        print(f"  {k}: {v}")

print("\n=== text_config（数値/文字列のみ）===")
for k in sorted(tc):
    v = tc[k]
    if isinstance(v, (dict, list)):
        print(f"  {k}: {type(v).__name__}({len(v)})")
    else:
        print(f"  {k}: {v}")

print("\n=== text_config の dict/list の中身 ===")
for k in sorted(tc):
    v = tc[k]
    if isinstance(v, list):
        print(f"  {k} ({len(v)}): {v[:8]}")
    elif isinstance(v, dict):
        print(f"  {k}: {json.dumps(v)[:300]}")

layers = tc.get("layer_types") or []
n = tc.get("num_hidden_layers")
print("\n=== 層種別 ===")
print("  num_hidden_layers:", n, "| layer_types:", len(layers),
      "| 一致:", n == len(layers))
if layers:
    from collections import Counter
    print("  内訳:", Counter(layers))
    full = [i for i, t in enumerate(layers) if "full" in t]
    print("  full_attention 層:", full)

heads = tc.get("num_attention_heads")
kv = tc.get("num_key_value_heads")
hd = tc.get("head_dim") or (tc.get("hidden_size", 0) // max(heads or 1, 1))
hidden = tc.get("hidden_size")
inter = tc.get("intermediate_size")
print("\n=== 形状 ===")
print(f"  hidden={hidden} heads={heads} kv_heads={kv} head_dim={hd} "
      f"intermediate={inter}")
print(f"  q_dim={heads * hd if heads and hd else None} "
      f"kv_dim={kv * hd if kv and hd else None}")

print("\n=== 層種別ごとの重み形状（safetensors header から）===")
import struct


def header(path):
    with open(path, "rb") as f:
        ln = struct.unpack("<Q", f.read(8))[0]
        return json.loads(f.read(ln).decode("utf-8", "replace")), 8 + ln


h, base = header(ROOT + "/model.safetensors")
FMT = {"I32": ("<2i", 8), "I64": ("<2q", 16), "U32": ("<2I", 8), "U64": ("<2Q", 16)}
shapes = {}
with open(ROOT + "/model.safetensors", "rb") as f:
    for k, v in h.items():
        if not k.endswith("weight_shape"):
            continue
        a, b = v["data_offsets"]
        fmt, need = FMT.get(v.get("dtype"), ("<2i", 8))
        f.seek(base + a)
        shapes[k] = struct.unpack(fmt, f.read(need))
print("  weight_shape dtype:", {h[k].get("dtype") for k in shapes},
      "| 件数:", len(shapes))
for idx in (0, 1, 2, 3):
    print(f"  --- layer {idx} ---")
    for k in sorted(shapes):
        if f".layers.{idx}." in k and k.endswith((
                "in_proj_qkv.weight_shape", "in_proj_z.weight_shape",
                "out_proj.weight_shape", "gate_up_proj.weight_shape",
                "down_proj.weight_shape", "q_proj.weight_shape",
                "o_proj.weight_shape")):
            print(f"    {k.split('.layers.')[-1]}: {shapes[k]}")

print("\n=== KV サイズ計算（実測 config から）===")
kv_bytes_per_tok = 2 * len(full) * kv * hd * 2 if kv and hd else None
print(f"  full 層 {len(full)} × kv_heads {kv} × head_dim {hd} × 2(K,V) × 2B "
      f"= {kv_bytes_per_tok} B/token = {(kv_bytes_per_tok or 0) / 1024:.1f} KiB")
for mlen in (8192, 16384, 32768, 65536):
    if kv_bytes_per_tok:
        per = kv_bytes_per_tok * mlen / 2 / 2**30
        print(f"    max_len={mlen:6d}: {per:.2f} GiB/GPU (TP=2, KV only)")

print("\n=== 線形注意の状態サイズ ===")
for k in sorted(tc):
    if any(x in k for x in ("conv", "ssm", "linear", "state", "chunk", "expand")):
        print(f"  {k}: {tc[k]}")

print("\n=== rope / 長さ上限 ===")
for k in sorted(tc):
    if "rope" in k or "max_position" in k or "sliding" in k:
        print(f"  {k}: {tc[k]}")
print("  トップレベル rope/max:", {k: cfg.get(k) for k in
                              ("max_position_embeddings", "rope_scaling",
                               "rope_theta") if k in cfg})
